# BOJ Policy Meeting OIS & Actual Rate EDA

このノートブックでは、日銀の**実績政策金利（無担保コール翌日物誘導目標）**と、将来の各会合での期待金利（M1〜M8）、および関連市場指標の可視化を行います。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set(style="whitegrid")
plt.rcParams['font.family'] = 'AppleGothic'

## 1. データの読み込みと統合
日銀の公式発表に基づいた政策金利ヒストリーと、市場のOISデータを統合します。

In [ ]:
file_path = 'data/BOJ_data.xlsx'
df_raw = pd.read_excel(file_path)
df = df_raw.iloc[1:].copy()
df['日付'] = pd.to_datetime(df['日付'], format='%Y年%m月%d日')
df = df.sort_values('日付').reset_index(drop=True)

cols_to_fix = df.columns.drop('日付')
for col in cols_to_fix:
    df[col] = pd.to_numeric(df[col], errors='coerce')

rename_dict = {
    'JPY1DOIS=ICAP (MID_PRICE)': '1D_OIS',
    'JPBOJ1ONI=TRDT (MID_PRICE)': 'M1', 
    'JPBOJ2ONI=TRDT (MID_PRICE)': 'M2',
    'JPBOJ3ONI=TRDT (MID_PRICE)': 'M3', 
    'JPBOJ4ONI=TRDT (MID_PRICE)': 'M4',
    'JPBOJ5ONI=TRDT (MID_PRICE)': 'M5', 
    'JPBOJ6ONI=TRDT (MID_PRICE)': 'M6',
    'JPBOJ7ONI=TRDT (MID_PRICE)': 'M7', 
    'JPBOJ8ONI=TRDT (MID_PRICE)': 'M8',
    'JPY= (MID_PRICE)': 'USDJPY', 
    'JGBc1 (TRDPRC_1)': 'JGB_Future',
    '.N225 (TRDPRC_1)': 'Nikkei225', 
    '.DXY (TRDPRC_1)': 'DXY'
}
df = df.rename(columns=rename_dict)

# 会合データのマージと実績金利の埋め合わせ
df_meetings = pd.read_csv('data/BOJ_meeting_history.csv')
df_meetings['Date'] = pd.to_datetime(df_meetings['Date'])
df = pd.merge(df, df_meetings, left_on='日付', right_on='Date', how='left')
df['Actual_Policy_Rate'] = df['Policy_Rate'].ffill()

boj_rates = ['1D_OIS', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8']
external_indices = ['USDJPY', 'JGB_Future', 'Nikkei225', 'DXY']
df.head()

## 2. 政策金利実績 vs 会合期待金利の推移
ステップ状に変化する日銀の実績金利と、市場の期待値がどのように先行して動いているかを比較します。

In [ ]:
plt.figure(figsize=(15, 8))
# 実績金利 (ステップ状)
plt.step(df['日付'], df['Actual_Policy_Rate'], where='post', color='black', lw=4, label='Actual Policy Rate', alpha=0.8)

# 市場の期待金利 (M1, M3, M5, M8)
for r in ['M1', 'M3', 'M5', 'M8']:
    plt.plot(df['日付'], df[r], label=f'Expected {r}', alpha=0.6, linestyle='--')

plt.title('BOJ Actual Policy Rate vs Market Expectations (OIS)', fontsize=16)
plt.ylabel('Rate (%)')
plt.legend(loc='upper left', frameon=True)
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

## 3. 会合パスの形状変化
現在の政策金利を起点とした、将来の会合ごとの利上げ織り込み度合いを確認。

In [ ]:
plt.figure(figsize=(10, 6))
latest_idx = df.index[-1]
prev_idx = df.index[-60] # 約3ヶ月前

x_labels = ['Current (Actual)', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8']
current_actual = df.loc[latest_idx, 'Actual_Policy_Rate']
expected_path = [current_actual] + df.loc[latest_idx, ['M1','M2','M3','M4','M5','M6','M7','M8']].tolist()

prev_actual = df.loc[prev_idx, 'Actual_Policy_Rate']
prev_expected_path = [prev_actual] + df.loc[prev_idx, ['M1','M2','M3','M4','M5','M6','M7','M8']].tolist()

plt.plot(range(len(expected_path)), expected_path, marker='o', lw=3, label=f'Latest ({df["日付"].iloc[-1].strftime("%Y-%m-%d")})')
plt.plot(range(len(prev_expected_path)), prev_expected_path, marker='o', lw=3, linestyle='--', label=f'3 Months Ago ({df["日付"].iloc[-60].strftime("%Y-%m-%d")})')

plt.title('BOJ Rate Hike Path: From Actual Rate to Future Meetings', fontsize=14)
plt.xlabel('Meeting Steps (Future)')
plt.ylabel('Rate (%)')
plt.xticks(range(len(expected_path)), x_labels)
plt.legend()
plt.grid(axis='y', linestyle=':', alpha=0.7)
plt.show()

## 4. 外部指標（為替・株）との関係性
利上げ期待の強まり（M1 - Actual Rate）がドル円や日経平均にどう影響しているかを確認します。

In [ ]:
df['M1_Surprise'] = df['M1'] - df['Actual_Policy_Rate']

fig, ax1 = plt.subplots(figsize=(14, 7))
ax1.plot(df['日付'], df['M1_Surprise'], color='purple', label='M1 Surprise (M1 - Actual)', alpha=0.7)
ax1.set_ylabel('Spread (%)')
ax1.legend(loc='upper left')

ax2 = ax1.twinx()
ax2.plot(df['日付'], df['USDJPY'], color='blue', label='USDJPY', alpha=0.5)
ax2.set_ylabel('USDJPY')
ax2.legend(loc='upper right')

plt.title('M1 Expectation Surprise vs USDJPY', fontsize=14)
plt.show()